# DLP Term Project: Does a Fairness Fix Have to Change CLIP? Adversarial Training vs. Fine-Tuning for Child/Adult Detection

---

**Due date:** 2026.12.17 23:59

**Submission:** `202555404_IsaacMutuma.ipynb`, `202555404_IsaacMutuma_Report.pdf` and a 10-minute video.

**Track:** C (free topic), linked to Track A (ViT fine-tuning) and Track B (adversarial training).

<h2><span style="color:blue">[202555404] [Isaac Mutuma]</span> </h2>

| Step | What it does |
|---|---|
| **0. Setup** | Packages, GPU check, which parts to run |
| **1. Data** | Downloads UTKFace (pinned version, checked by SHA-256) and reads age and race from the file names |
| **2. CLIP features** | Pinned CLIP ViT-L/14 (OpenAI weights, QuickGELU checked), one 768-number embedding per image |
| **3. Folds** | 5 fixed folds (seed 0), balanced by race and age |
| **4. Evaluation code** | Detection rates, the White − Black gap, paired bootstrap, Holm |
| **5. Model code** | Age head (A), adversarial training (E), leakage probe, LEACE: *my implementation* |
| **6. Experiments** | A and E, each started with the three sentences |
| **7. Fine-tuning (F1-1, F1-4, F2)** | LoRA inside CLIP |
| **8. Re-check** | Main comparison with 3 seeds |
| **9. Follow-up questions** | Two questions that came from my results |
| **10. Visual cases and demo** | A few clearly harmless ICCWD photos run through the models |
| **11. Limits and conclusion** | |
| **12. Experiment log** | Settings, epochs, updates, seeds and run time of every run |

**Ethics.** No images of children are generated. UTKFace photos are only read by the model and never shown. Real photos appear only in Step 10: a few clearly harmless ICCWD scenes, shown as they are; no captions or links.

## 0. Setup

Runs on Colab (Runtime → Change runtime type → T4) or in Jupyter on the course GPU server (set `GPU_ID` below to your assigned GPU). Section 10 needs the ICCWD photos on Google Drive, so it runs on Colab. Approximate time on a T4: data and CLIP features 25 min, A 5 min, E 90 min, follow-up 1 60 min, follow-up 2 25 min. Set a part to `False` to skip it.

In [ ]:
import datetime
print("This code is written at " + str(datetime.datetime.now()))

In [ ]:
!pip -q install open_clip_torch==3.3.0
GPU_ID = None   # course GPU server: set to your assigned GPU id, e.g. "2"; on Colab leave None
import os
if GPU_ID is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)   # must be set before torch starts
RUN = {"A": True, "E": True, "followup1": True, "followup2": True, "F": False, "recheck": False}
import os, json, math, hashlib, tarfile, re, time, urllib.request
from pathlib import Path, PurePosixPath
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
%matplotlib inline
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK = Path("/content/work") if Path("/content").exists() else Path("work")
WORK.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "",
      "| torch", torch.__version__)

In [ ]:
# Fixed settings (chosen before any model was trained; same for every condition)
from types import SimpleNamespace
C = SimpleNamespace(
    SEED=0, N_FOLDS=5, N_BOOTSTRAP=2000, ALPHA=0.05,
    CHILD_AGE_THRESHOLD=18,                 # flagged as a child if predicted age < 18
    MINOR_MAX_AGE=17, ADULT_FPR_RANGE=(20, 59),
    MINOR_STRATA=[(0, 9), (10, 17)],
    STRATUM_WEIGHTS={"0-9": 0.7234, "10-17": 0.2766},   # same age weights for every group
    FOLD_AGE_BANDS=[(0, 9), (10, 17), (18, 19), (20, 39), (40, 59), (60, 200)],
    UTKFACE_RACES={0: "White", 1: "Black", 2: "Asian", 3: "Indian", 4: "Others"},
    TESTED_GROUPS=["White", "Black", "Asian", "Indian"],  # "Others" is trained on, not reported
    PRIMARY_CONTRAST=("White", "Black"),
    SUCCESS={"small_gap_max": 0.03, "adult_fpr_max": 0.05, "adult_fpr_vs_A": 0.02,
             "mae_noninferiority_years": 0.5},
    TUNING={"tpr_drop_max": 0.02},
    HOLM_FAMILY=["E_inv", "F1_1", "F1_4", "F2"],
)
UTKFACE = {"hf_repo": "py97/UTKFace-Cropped", "revision": "e80f21fb21fa3631380c3cb634b79be3ac2f4cba",
           "file": "UTKFace.tar.gz",
           "sha256": "2c0655397b498e81b82d685ee74bee8c2e4364a5897caf3afec6ea48929024a7",
           "n_records": 23705}
CLIP_ARCH = "ViT-L-14"
CLIP_WEIGHTS = {"hf_repo": "timm/vit_large_patch14_clip_224.openai",
                "revision": "18d0535469bb561bf468d76c1d73aa35156c922b",
                "file": "open_clip_model.safetensors",
                "sha256": "9ce2e8a8ebfff3793d7d375ad6d3c35cb9aebf3de7ace0fc7308accab7cd207e"}

## 1. Data: UTKFace

23,705 face photos. Age, gender and race come from the file name `age_gender_race_date.jpg`. Three files have malformed names and are skipped. The photos are never displayed.

In [ ]:
def sha256(path, bs=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(bs), b""):
            h.update(b)
    return h.hexdigest()

from huggingface_hub import hf_hub_download
tar_path = hf_hub_download(UTKFACE["hf_repo"], UTKFACE["file"], revision=UTKFACE["revision"], repo_type="dataset")
assert sha256(tar_path) == UTKFACE["sha256"], "UTKFace archive does not match the pinned version"
img_dir = WORK / "UTKFace"
if not img_dir.exists():
    with tarfile.open(tar_path) as t:
        safe = [m for m in t.getmembers() if m.isfile() and ".." not in PurePosixPath(m.name).parts]
        t.extractall(WORK, members=safe)
    cand = [p for p in WORK.iterdir() if p.is_dir() and p.name.lower().startswith("utkface") and p != img_dir]
    if cand: cand[0].rename(img_dir)

NAME = re.compile(r"^(\d+)_([01])_([0-4])_(\d+)")
rows = []
for p in sorted(img_dir.iterdir()):
    m = NAME.match(p.name)
    if m and p.suffix.lower() in (".jpg", ".jpeg", ".png"):
        rows.append({"id": p.name.split(".")[0], "age": float(m[1]),
                     "group": C.UTKFACE_RACES[int(m[3])], "path": str(p)})
df = pd.DataFrame(rows).sort_values("id").reset_index(drop=True)
assert len(df) == UTKFACE["n_records"], len(df)
df["band"] = pd.cut(df.age, [-1, 9, 17, 19, 39, 59, 200], labels=["0-9", "10-17", "18-19", "20-39", "40-59", "60+"])
display(pd.crosstab(df.group, df.band, margins=True))

## 2. CLIP ViT-L/14 features

The OpenAI weights were trained with QuickGELU; loading them from a file makes `open_clip` silently use plain GELU, so I force QuickGELU and check that no GELU layer is left. Each image becomes one 768-number embedding (before L2 normalisation). The result is cached in `WORK`.

In [ ]:
import open_clip
from PIL import Image

def load_clip(device):
    path = hf_hub_download(CLIP_WEIGHTS["hf_repo"], CLIP_WEIGHTS["file"], revision=CLIP_WEIGHTS["revision"])
    assert sha256(path) == CLIP_WEIGHTS["sha256"], "CLIP weights do not match the pinned version"
    model, _, preprocess = open_clip.create_model_and_transforms(CLIP_ARCH, pretrained=path, force_quick_gelu=True)
    names = [type(m).__name__ for m in model.modules()]
    assert names.count("QuickGELU") > 0 and names.count("GELU") == 0, "CLIP must use QuickGELU"
    return model.eval().to(device), preprocess

emb_file = WORK / "utkface_clip_vitl14.npz"
if emb_file.exists():
    z = np.load(emb_file, allow_pickle=False)
    assert list(z["ids"]) == list(df.id)
    X = z["emb"]
else:
    clip_model, preprocess = load_clip(DEVICE)
    class DS(torch.utils.data.Dataset):
        def __len__(self): return len(df)
        def __getitem__(self, i): return preprocess(Image.open(df.path[i]).convert("RGB"))
    out, t0 = [], time.time()
    with torch.no_grad():
        for x in torch.utils.data.DataLoader(DS(), batch_size=128, num_workers=2):
            out.append(clip_model.encode_image(x.to(DEVICE)).float().cpu().numpy())
    X = np.concatenate(out).astype(np.float32)
    np.savez(emb_file, ids=df.id.values, emb=X)
    print(f"embedded {len(X)} images in {time.time() - t0:.0f} s")
ages, groups = df.age.values.astype(float), df.group.values
print("embeddings:", X.shape, "| sha256", sha256(emb_file)[:16])

## 3. Folds

Stratified 5-fold split over race × age band (seed 0, ids sorted). Every image is predicted by a model that never saw it. For test fold *k*, settings are tuned on fold (*k*+1) mod 5 and the model is refit on the other four folds.

In [ ]:
from collections import Counter

import numpy as np



def age_band(age, bands=None):
    """Label like '10-17' of the band holding `age` ('60+' for the last band)."""
    for lo, hi in bands or C.FOLD_AGE_BANDS:
        if lo <= age <= hi:
            return f"{lo}+" if hi >= 200 else f"{lo}-{hi}"
    raise ValueError(f"age {age} outside all bands")


def strat_keys(groups, ages, k):
    """Stratification key per image, with small cells merged (see module docstring)."""
    keys = [f"{g}|{age_band(a)}" for g, a in zip(groups, ages)]
    cnt = Counter(keys)
    keys = [x if cnt[x] >= k else f"{x.split('|')[0]}|rare" for x in keys]
    cnt = Counter(keys)
    keys = [x if cnt[x] >= k else "rare" for x in keys]
    if 0 < Counter(keys)["rare"] < k:   # pool still too small: join the group's largest cell
        cnt = Counter(keys)
        biggest = {}
        for x in keys:
            g = x.split("|")[0]
            if x != "rare" and cnt[x] > cnt.get(biggest.get(g), 0):
                biggest[g] = x
        top = max(cnt, key=cnt.get)
        keys = [biggest.get(g.split("|")[0], top) if x == "rare" else x
                for x, g in zip(keys, [f"{gg}|" for gg in groups])]
    return keys


def make_folds(ids, groups, ages, k=None, seed=None):
    """dict id -> fold (0..k-1). Deterministic given the same (id, group, age) set."""
    from sklearn.model_selection import StratifiedKFold
    k = k or C.N_FOLDS
    seed = C.SEED if seed is None else seed
    order = np.argsort(np.asarray(ids, dtype=object), kind="stable")
    ids_s = [ids[i] for i in order]
    keys = strat_keys([groups[i] for i in order], [ages[i] for i in order], k)
    if len(set(ids_s)) != len(ids_s):
        raise ValueError("duplicate ids")
    fold = np.empty(len(ids_s), dtype=int)
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    for f, (_, test) in enumerate(skf.split(np.zeros(len(ids_s)), keys)):
        fold[test] = f
    return dict(zip(ids_s, fold.tolist()))


def inner_val_fold(test_fold, k=None):
    """Inner validation fold used for tuning when `test_fold` is held out."""
    return (test_fold + 1) % (k or C.N_FOLDS)

fold_of = make_folds(list(df.id), list(groups), list(ages))
folds = np.array([fold_of[i] for i in df.id])
display(pd.crosstab(folds, groups))

## 4. Evaluation code

Age-stratified detection rate (TPR) of minors with the same age weights for every group, adult false-positive rate (FPR, ages 20–59), age error, and the White − Black gap. Confidence intervals: 2,000 bootstrap resamples within each group; every model is scored on the same resamples, so comparisons with model A are paired. Ported from my earlier audit.

In [ ]:
import numpy as np
import pandas as pd


STRATA = list(C.STRATUM_WEIGHTS)                     # ["0-9", "10-17"]


# ── data ─────────────────────────────────────────────────────────────────────
def prepare(df):
    """Sorted copy with derived columns: flagged, minor, teen, adult, stratum, abs_err."""
    need = {"id", "group", "age", "pred_age"}
    if not need <= set(df.columns):
        raise ValueError(f"missing columns {need - set(df.columns)}")
    if df["id"].duplicated().any():
        raise ValueError("duplicate ids")
    if df["pred_age"].isna().any():
        raise ValueError("missing predictions")
    d = df.sort_values("id", kind="stable").reset_index(drop=True).copy()
    lo, hi = C.ADULT_FPR_RANGE
    d["flagged"] = (d.pred_age < C.CHILD_AGE_THRESHOLD).astype(float)
    d["minor"] = d.age <= C.MINOR_MAX_AGE
    d["teen"] = (d.age >= 10) & d.minor
    d["adult"] = (d.age >= lo) & (d.age <= hi)
    d["stratum"] = np.where(d.age <= 9, "0-9", np.where(d.minor, "10-17", ""))
    d["err"] = d.pred_age - d.age
    d["abs_err"] = d.err.abs()
    return d


class BootPlan:
    """Within-group multinomial resample counts, shared by every model (pairing).

    W[g] has shape (B, n_g): how often each image of group g is drawn in each replicate.
    """

    def __init__(self, d, B=None, seed=None):
        self.B = B or C.N_BOOTSTRAP
        rng = np.random.default_rng(C.SEED if seed is None else seed)
        self.ids = d["id"].to_numpy()
        self.idx, self.W = {}, {}
        for g in sorted(d.group.unique()):
            idx = np.flatnonzero(d.group.to_numpy() == g)
            n = len(idx)
            self.idx[g] = idx
            self.W[g] = rng.multinomial(n, np.full(n, 1.0 / n), size=self.B).astype(np.float32)

    def check(self, d):
        """Raise unless `d` (prepared) has exactly the images this plan was built on."""
        if not np.array_equal(d["id"].to_numpy(), self.ids):
            raise ValueError("model predictions cover a different image set than the plan")


# ── estimates: point value + bootstrap replicates ────────────────────────────
def _mean(d, plan, g, values, mask):
    """(point, reps) for mean(values | mask) within group g."""
    i = plan.idx[g]
    v = np.asarray(values, float)[i]
    m = np.asarray(mask, float)[i]
    if m.sum() == 0:
        return float("nan"), np.full(plan.B, np.nan)
    W = plan.W[g]
    with np.errstate(invalid="ignore", divide="ignore"):
        reps = (W @ (v * m)) / (W @ m)
    return float((v * m).sum() / m.sum()), reps


def _strat(d, plan, g, values):
    """(point, reps) for the stratified minor mean of `values` within group g."""
    pt, reps = 0.0, np.zeros(plan.B)
    for s, w in C.STRATUM_WEIGHTS.items():
        p, r = _mean(d, plan, g, values, d.stratum.to_numpy() == s)
        pt, reps = pt + w * p, reps + w * r
    return pt, reps


def ci(reps):
    """Percentile 95% interval of bootstrap replicates (NaNs ignored)."""
    reps = np.asarray(reps, float)
    reps = reps[~np.isnan(reps)]
    if len(reps) == 0:
        return [None, None]
    a = C.ALPHA / 2
    return [float(np.quantile(reps, a)), float(np.quantile(reps, 1 - a))]


def _block(pt, reps):
    return {"est": None if np.isnan(pt) else float(pt), "ci": ci(reps)}


def estimates(d, plan):
    """All point estimates + replicates for one model, keyed by metric name."""
    plan.check(d)
    E = {}
    for g in plan.idx:
        E[f"tpr_strat|{g}"] = _strat(d, plan, g, d.flagged)
        E[f"tpr_teen|{g}"] = _mean(d, plan, g, d.flagged, d.teen)
        E[f"tpr_0_9|{g}"] = _mean(d, plan, g, d.flagged, d.stratum == "0-9")
        E[f"fpr_adult|{g}"] = _mean(d, plan, g, d.flagged, d.adult)
        E[f"age_err_minor_strat|{g}"] = _strat(d, plan, g, d.err)
        E[f"mae|{g}"] = _mean(d, plan, g, d.abs_err, np.ones(len(d), bool))
    a, b = C.PRIMARY_CONTRAST
    E["tpr_gap_W_minus_B"] = tuple(x - y for x, y in zip(E[f"tpr_strat|{a}"], E[f"tpr_strat|{b}"]))
    E["tpr_teen_gap_W_minus_B"] = tuple(x - y for x, y in
                                        zip(E[f"tpr_teen|{a}"], E[f"tpr_teen|{b}"]))
    E["fpr_diff_B_minus_W"] = tuple(x - y for x, y in zip(E[f"fpr_adult|{b}"], E[f"fpr_adult|{a}"]))
    E["age_err_gap_B_minus_W"] = tuple(x - y for x, y in zip(E[f"age_err_minor_strat|{b}"],
                                                             E[f"age_err_minor_strat|{a}"]))
    tested = [g for g in C.TESTED_GROUPS if g in plan.idx]
    E["worst_group_tpr_strat"] = (min(E[f"tpr_strat|{g}"][0] for g in tested),
                                  np.min([E[f"tpr_strat|{g}"][1] for g in tested], axis=0))
    # overall MAE over ALL images (incl. Others): group means weighted by group size,
    # so each replicate keeps the observed group sizes (resampling is within group)
    n = {g: len(i) for g, i in plan.idx.items()}
    N = sum(n.values())
    E["mae_overall"] = (sum(n[g] * E[f"mae|{g}"][0] for g in n) / N,
                        sum(n[g] * E[f"mae|{g}"][1] for g in n) / N)
    return E


def model_report(d, plan):
    """JSON-ready report for one model: per tested group + contrasts, with CIs."""
    E = estimates(d, plan)
    groups = {}
    for g in C.TESTED_GROUPS:
        if g not in plan.idx:
            continue
        gd = d.iloc[plan.idx[g]]
        groups[g] = {"n": int(len(gd)), "n_minor": int(gd.minor.sum()),
                     "n_teen_10_17": int(gd.teen.sum()), "n_adult_20_59": int(gd.adult.sum()),
                     **{k.split("|")[0]: _block(*v) for k, v in E.items() if k.endswith(f"|{g}")}}
    keys = ["tpr_gap_W_minus_B", "tpr_teen_gap_W_minus_B", "fpr_diff_B_minus_W",
            "age_err_gap_B_minus_W", "worst_group_tpr_strat", "mae_overall"]
    return {"by_group": groups, **{k: _block(*E[k]) for k in keys},
            "n_images": int(len(d)), "bootstrap": {"B": plan.B, "seed": C.SEED,
                                                   "scheme": "within group, multinomial"}}


# ── paired comparison with model A and the success rule ──────────────────────
def paired_vs_A(dA, dM, plan):
    """Paired bootstrap of (gap_A - gap_M) and (MAE_M - MAE_A) on the same resamples."""
    for col in ("group", "age"):
        if not np.array_equal(dA[col].to_numpy(), dM[col].to_numpy()):
            raise ValueError(f"{col} differs between models")
    EA, EM = estimates(dA, plan), estimates(dM, plan)
    gA, gM = EA["tpr_gap_W_minus_B"], EM["tpr_gap_W_minus_B"]
    diff_pt, diff_reps = gA[0] - gM[0], gA[1] - gM[1]
    p = (np.sum(diff_reps <= 0) + 1) / (plan.B + 1)
    mae_pt, mae_reps = EM["mae_overall"][0] - EA["mae_overall"][0], \
        EM["mae_overall"][1] - EA["mae_overall"][1]
    return {"gap_reduction_A_minus_M": {"est": float(diff_pt), "ci": ci(diff_reps),
                                        "p_one_sided": float(p)},
            "mae_increase_M_minus_A": {"est": float(mae_pt), "ci": ci(mae_reps)}}


def holm(pvals):
    """Holm step-down adjusted p-values for a dict name -> p."""
    keys = list(pvals)
    ps = np.array([pvals[k] for k in keys], float)
    m, adj, running = len(ps), np.zeros(len(ps)), 0.0
    for rank, i in enumerate(np.argsort(ps, kind="stable")):
        running = max(running, min(1.0, (m - rank) * ps[i]))
        adj[i] = running
    return {k: float(adj[i]) for i, k in enumerate(keys)}


def verdict(rep_A, rep_M, paired, p_holm):
    """Apply SUCCESS_CRITERIA.md §4. Returns each criterion and 'closes'/'narrows'/'neither'."""
    S = C.SUCCESS
    gapA, gapM = rep_A["tpr_gap_W_minus_B"], rep_M["tpr_gap_W_minus_B"]
    c1 = bool(p_holm < C.ALPHA)
    c2 = bool(gapM["est"] <= S["small_gap_max"] and gapM["ci"][1] < gapA["est"])
    fpr = {g: (rep_M["by_group"][g]["fpr_adult"]["est"], rep_A["by_group"][g]["fpr_adult"]["est"])
           for g in rep_M["by_group"]}
    c3_by = {g: bool(m <= S["adult_fpr_max"] and m <= a + S["adult_fpr_vs_A"])
             for g, (m, a) in fpr.items()}
    c3 = all(c3_by.values())
    c4 = bool(paired["mae_increase_M_minus_A"]["ci"][1] <= S["mae_noninferiority_years"])
    outcome = "closes" if (c1 and c2 and c3 and c4) else \
        "narrows" if (c1 and c3 and c4) else "neither"
    return {"c1_fairer_holm": c1, "p_holm": float(p_holm), "c2_small_gap": c2,
            "c3_adult_fpr": c3, "c3_by_group": c3_by, "c4_mae_noninferior": c4,
            "outcome": outcome}


def a_self_check(rep_A):
    """§4 'Model A itself': does A's own gap CI already contain 0?"""
    lo, hi = rep_A["tpr_gap_W_minus_B"]["ci"]
    return {"gap_ci_contains_0": bool(lo <= 0 <= hi), "gap": rep_A["tpr_gap_W_minus_B"]}


def frame(ids, groups, ages, preds):
    """Convenience: DataFrame in the format prepare() expects."""
    return pd.DataFrame({"id": ids, "group": groups, "age": np.asarray(ages, float),
                         "pred_age": np.asarray(preds, float)})

### Leakage probe (my implementation)

After training, a new logistic regression tries to read race from a representation. Score: balanced accuracy over the 5 groups (chance 0.20).

In [ ]:
import numpy as np



def fit_predict_probe(Ztr, gtr, Zte, seed=0):
    """Race predictions on Zte from a logistic regression fitted on (Ztr, gtr)."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler().fit(Ztr)
    clf = LogisticRegression(C=1.0, max_iter=3000, random_state=seed)
    clf.fit(sc.transform(Ztr), np.asarray(gtr))
    return clf.predict(sc.transform(Zte))


def balanced_accuracy(pred, true, classes=None):
    """Mean per-class recall."""
    pred, true = np.asarray(pred), np.asarray(true)
    classes = sorted(set(true.tolist())) if classes is None else classes
    return float(np.mean([np.mean(pred[true == c] == c) for c in classes]))


def summarise(probe_out, groups, B=None, seed=None):
    """Pooled balanced accuracy per representation with paired bootstrap CIs."""
    B = C.N_BOOTSTRAP if B is None else B
    seed = C.SEED if seed is None else seed
    groups = np.asarray(groups)
    classes = sorted(set(groups.tolist()))
    names = list(probe_out)
    for n in names:
        if any(v is None for v in probe_out[n]):
            raise ValueError(f"probe '{n}' is missing predictions")
    rng = np.random.default_rng(seed)
    # per class: (B, n_reps) resampled recall
    rec_boot = np.zeros((len(classes), B, len(names)))
    point = {n: {} for n in names}
    for ci, c in enumerate(classes):
        idx = np.flatnonzero(groups == c)
        hit = np.stack([(np.asarray(probe_out[n])[idx] == c).astype(float) for n in names], 1)
        for j, n in enumerate(names):
            point[n][c] = float(hit[:, j].mean())
        W = rng.multinomial(len(idx), np.full(len(idx), 1 / len(idx)), size=B)
        rec_boot[ci] = (W @ hit) / len(idx)
    bal_boot = rec_boot.mean(0)                      # (B, n_reps)
    lo, hi = 100 * C.ALPHA / 2, 100 * (1 - C.ALPHA / 2)
    out = {"chance": 1 / len(classes), "classes": classes, "n": int(len(groups)),
           "majority_share": float(max(np.mean(groups == c) for c in classes)),
           "bal_acc": {}, "recall_by_race": point, "diff": {}}
    for j, n in enumerate(names):
        est = float(np.mean(list(point[n].values())))
        out["bal_acc"][n] = {"est": est, "ci": np.percentile(bal_boot[:, j], [lo, hi]).tolist()}
    if "model" in names:
        jm = names.index("model")
        for n in names:
            if n == "model":
                continue
            d = bal_boot[:, jm] - bal_boot[:, names.index(n)]
            out["diff"][f"model_minus_{n}"] = {
                "est": out["bal_acc"]["model"]["est"] - out["bal_acc"][n]["est"],
                "ci": np.percentile(d, [lo, hi]).tolist()}
    return out

## 5. Model code (my implementation)

In [ ]:
# ── Age head, model E and the out-of-fold loop (my implementation) ───────────
import itertools, time
import numpy as np
import torch
import torch.nn as nn

GRID_A = {"hidden": [0, 512], "lr": [1e-3, 3e-4], "weight_decay": [1e-4, 1e-2]}
FIXED_A = {"max_epochs": 60, "batch_size": 256, "dropout": 0.1, "loss": "l1", "method": "plain"}
GRID_E = {"hidden": [512], "lr": [1e-3, 3e-4], "weight_decay": [1e-4, 1e-2],
          "adv_lambda": [0.1, 0.3, 1.0, 3.0], "adv_target": ["max_ce", "prior"]}
FIXED_E = {**FIXED_A, "method": "adv_inv", "adv_hidden": 128, "adv_lr": 1e-2, "adv_steps": 5,
           "adv_weight_decay": 1e-4, "leakage_probe": True}


def grid_settings(grid):
    keys = list(grid)
    return [dict(zip(keys, v)) for v in itertools.product(*(grid[k] for k in keys))]


def _l2(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)


class Standardiser:
    """L2-normalise, then z-score with statistics from the training rows only."""

    def fit(self, X):
        Z = _l2(X)
        self.mu, self.sd = Z.mean(0), Z.std(0) + 1e-6
        return self

    def __call__(self, X):
        return ((_l2(X) - self.mu) / self.sd).astype(np.float32)


def make_head(dim, hidden, dropout):
    """Age head: linear, or MLP = Linear + GELU (= z, 512 numbers) -> Dropout -> Linear."""
    if hidden == 0:
        return nn.Linear(dim, 1)
    return nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, 1))


def make_adversary(dim, hidden, n_classes):
    """Race classifier on z (model E)."""
    return nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Linear(hidden, n_classes))


def predict(model, X):
    model.eval()
    dev = next(model.parameters()).device
    with torch.no_grad():
        Xt = X if torch.is_tensor(X) else torch.from_numpy(X)
        return model(Xt.to(dev)).squeeze(1).cpu().numpy().astype(float)


def encode(model, X):
    """z = output of the MLP's hidden layer."""
    model.eval()
    dev = next(model.parameters()).device
    out = []
    with torch.no_grad():
        for b in range(0, len(X), 4096):
            out.append(model[:2](torch.from_numpy(np.ascontiguousarray(X[b:b + 4096])).to(dev)).cpu().numpy())
    return np.concatenate(out).astype(np.float32)


def fit_head(Xtr, ytr, s, epochs, seed, device, Xval=None, gtr=None, atr=None):
    """Train one age head. Returns (predict_fn, per-epoch validation predictions, info).

    method 'plain' (model A): L1 on the standardised age.
    method 'adv_inv' (model E): per batch, (1) adv_steps updates of the race adversary on
    z with the MLP fixed, then (2) one update of the MLP on
    L1(age) - lambda * CE(race | z)  [target 'max_ce']  or
    L1(age) + lambda * KL(race base rates || adversary's guess)  [target 'prior'].
    lambda = 0 gives exactly model A (the adversary uses its own random-number stream).
    """
    method = s.get("method", "plain")
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = make_head(Xtr.shape[1], s["hidden"], s["dropout"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=s["lr"], weight_decay=s["weight_decay"])
    loss_fn = nn.L1Loss()
    y_mu, y_sd = float(np.mean(ytr)), float(np.std(ytr)) + 1e-6
    X = torch.from_numpy(Xtr).to(device)
    y = torch.from_numpy(((ytr - y_mu) / y_sd).astype(np.float32)).to(device)

    def predict_fn(Xn):
        return predict(model, Xn) * y_sd + y_mu

    if s["hidden"] > 0:
        predict_fn.encode = lambda Xn: encode(model, Xn)
    predict_fn.model, predict_fn.y_mu, predict_fn.y_sd = model, y_mu, y_sd   # model F starts from A's head
    info = {"method": method}
    if method == "adv_inv":
        classes = sorted(set(np.asarray(gtr).tolist()))
        rid = torch.from_numpy(np.searchsorted(classes, np.asarray(gtr))).to(device)
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(seed + 7919)
            adv = make_adversary(s["hidden"], s["adv_hidden"], len(classes))
        adv = adv.to(device)
        opt_adv = torch.optim.AdamW(adv.parameters(), lr=s["adv_lr"], weight_decay=s["adv_weight_decay"])
        ce_fn = nn.CrossEntropyLoss()
        lam, n_adv, target = float(s["adv_lambda"]), int(s["adv_steps"]), s.get("adv_target", "max_ce")
        log_prior = torch.log(torch.bincount(rid, minlength=len(classes)).float() / len(rid))
        enc, head = model[:2], model[2:]
        nC = len(classes)
        hist = {"adv_bal_acc": [], "adv_ce": []}

    Xv = torch.from_numpy(Xval).to(device) if Xval is not None else None
    val_preds = [] if Xval is not None else None
    for _ in range(epochs):
        model.train()
        order = torch.randperm(len(X), generator=g)
        if method == "adv_inv":
            conf = torch.zeros(nC, nC, dtype=torch.long, device=device)
            ce_sum = 0.0
        for b in order.split(s["batch_size"]):
            b = b.to(device)
            if method == "adv_inv":
                zb = enc(X[b])
                zd = zb.detach()
                for _ in range(n_adv):                    # (1) adversary, MLP fixed
                    opt_adv.zero_grad()
                    ce_fn(adv(zd), rid[b]).backward()
                    opt_adv.step()
                opt.zero_grad()                           # (2) MLP
                out = head(zb).squeeze(1)
                logits = adv(zb)
                ce = ce_fn(logits, rid[b])
                loss = loss_fn(out, y[b])
                if lam != 0 and target == "max_ce":
                    loss = loss - lam * ce
                elif lam != 0:
                    logp = torch.log_softmax(logits, 1)
                    loss = loss + lam * (log_prior.exp() * (log_prior - logp)).sum(1).mean()
                loss.backward()
                opt.step()
                with torch.no_grad():
                    conf += torch.bincount(rid[b] * nC + logits.argmax(1), minlength=nC * nC).view(nC, nC)
                    ce_sum += float(ce) * len(b)
                continue
            opt.zero_grad()
            out = model(X[b]).squeeze(1)
            loss_fn(out, y[b]).backward()
            opt.step()
        if method == "adv_inv":
            c = conf.cpu().numpy()
            hist["adv_bal_acc"].append(float(np.mean(np.diag(c) / np.maximum(c.sum(1), 1))))
            hist["adv_ce"].append(ce_sum / len(X))
        if Xv is not None:
            val_preds.append(predict_fn(Xv))
    if method == "adv_inv":
        info.update({"adv_classes": classes, "adv_chance_bal_acc": 1.0 / nC,
                     "adv_bal_acc_by_epoch": hist["adv_bal_acc"], "adv_ce_by_epoch": hist["adv_ce"]})
    return predict_fn, (np.stack(val_preds) if val_preds is not None else None), info


# ── tuning rule on the validation fold ───────────────────────────────────────
def selection_stats(pred, y, groups, per_group=False):
    pred, y, groups = np.asarray(pred), np.asarray(y), np.asarray(groups)
    flag = pred < C.CHILD_AGE_THRESHOLD
    lo, hi = C.ADULT_FPR_RANGE
    tprs, fprs = {}, {}
    for g in C.TESTED_GROUPS:
        m = groups == g
        t = 0.0
        for (a, b), w in zip(C.MINOR_STRATA, C.STRATUM_WEIGHTS.values()):
            s = m & (y >= a) & (y <= b)
            t += w * flag[s].mean() if s.any() else float("nan")
        tprs[g] = t
        ad = m & (y >= lo) & (y <= hi)
        fprs[g] = float(flag[ad].mean()) if ad.any() else float("nan")
    out = {"mae": float(np.abs(pred - y).mean()),
           "worst_group_tpr": float(np.nanmin(list(tprs.values()))),
           "max_adult_fpr": float(np.nanmax(list(fprs.values()))),
           "feasible": bool(np.nanmax(list(fprs.values())) <= C.SUCCESS["adult_fpr_max"])}
    if per_group:
        out["tpr_by_group"] = {g: float(t) for g, t in tprs.items()}
    return out


def group_tpr(pred, y, groups):
    return selection_stats(pred, y, groups, per_group=True)["tpr_by_group"]


def group_fpr_mae(pred, y, groups):
    pred, y, groups = np.asarray(pred), np.asarray(y), np.asarray(groups)
    flag = pred < C.CHILD_AGE_THRESHOLD
    lo, hi = C.ADULT_FPR_RANGE
    fpr = {}
    for g in C.TESTED_GROUPS:
        ad = (groups == g) & (y >= lo) & (y <= hi)
        fpr[g] = float(flag[ad].mean()) if ad.any() else float("nan")
    return {"fpr": fpr, "mae": float(np.abs(pred - y).mean())}


def selection_stats_v3(pred, y, groups, ref):
    """A setting is allowed only if, on the validation fold, every group's adult FPR is at
    most min(5%, A's + 2 points), MAE is at most A's + 0.5 years and no group's minor TPR
    falls more than 2 points below A's. Among allowed: best worst-group TPR, then lower MAE."""
    st = selection_stats(pred, y, groups)
    cur = group_fpr_mae(pred, y, groups)
    S = C.SUCCESS
    excess = [max(0.0, cur["fpr"][g] - min(S["adult_fpr_max"], ref["fpr"][g] + S["adult_fpr_vs_A"]))
              for g in C.TESTED_GROUPS]
    mae_excess = max(0.0, cur["mae"] - (ref["mae"] + S["mae_noninferiority_years"]))
    n_viol = sum(e > 0 for e in excess) + (mae_excess > 0)
    tpr = group_tpr(pred, y, groups)
    tpr_excess = [max(0.0, (ref["tpr"][g] - C.TUNING["tpr_drop_max"]) - tpr[g]) for g in C.TESTED_GROUPS]
    n_tpr = sum(e > 0 for e in tpr_excess)
    return {**st, "fpr_by_group": cur["fpr"], "tpr_by_group": tpr,
            "feasible": n_viol == 0 and n_tpr == 0, "n_violations": n_viol + n_tpr,
            "fpr_excess": float(sum(excess)), "tpr_excess": float(sum(tpr_excess)),
            "mae_excess": float(mae_excess)}


def rank_key(st, rule):
    if rule == "mae":
        return (st["mae"],)
    return (not st["feasible"], st["n_violations"], st["fpr_excess"], st["tpr_excess"],
            st["mae_excess"], -st["worst_group_tpr"], st["mae"])


def reference_A(rA, X, y, groups, tr_in, va, st, seed, k, device):
    """A's chosen setting for fold k, retrained on the same training folds (A's own seed)."""
    i = grid_settings(GRID_A).index({kk: rA["chosen"][kk] for kk in GRID_A})
    s = {**FIXED_A, **grid_settings(GRID_A)[i]}
    _, vp, _ = fit_head(st(X[tr_in]), y[tr_in], s, rA["epochs"], seed + 1000 * k + i, device, st(X[va]))
    ref = group_fpr_mae(vp[-1], y[va], groups[va])
    ref["tpr"] = group_tpr(vp[-1], y[va], groups[va])
    ref["mae_diff_vs_A_record"] = ref["mae"] - rA["inner_val_mae"]
    return ref


def leakage_fold(X, y, groups, tr_all, te, st, predict_fn, recA, seed, k, device, probe_out):
    """Race probe on z of this model, on z of model A, and on the CLIP inputs (test fold k)."""
    Xtr, Xte = st(X[tr_all]), st(X[te])
    pfA, _, _ = fit_head(Xtr, y[tr_all], {**FIXED_A, **recA["chosen"]}, recA["epochs"], seed + 1000 * k, device)
    reps = {"model": (predict_fn.encode(Xtr), predict_fn.encode(Xte)),
            "A": (pfA.encode(Xtr), pfA.encode(Xte)), "clip_raw": (Xtr, Xte)}
    out = {"A_refit_test_mae": float(np.abs(pfA(Xte) - y[te]).mean())}
    for name, (Ztr, Zte) in reps.items():
        p = fit_predict_probe(Ztr, groups[tr_all], Zte, seed)
        probe_out.setdefault(name, np.full(len(y), None, dtype=object))[te] = p
        out[f"bal_acc_{name}"] = balanced_accuracy(p, groups[te])
    return out


def oof_predictions(X, y, folds, grid, fixed, device="cpu", seed=None, log=print, groups=None,
                    select="mae", ref_A=None, probe_out=None, prep=None):
    """Out-of-fold predictions: for each test fold k, tune on folds != k, (k+1)%5 with
    validation fold (k+1)%5, refit on all four training folds, predict fold k."""
    seed = C.SEED if seed is None else seed
    K = int(folds.max()) + 1
    pred, record = np.full(len(y), np.nan), []
    for k in range(K):
        t0, v = time.time(), (k + 1) % K
        tr_in, va, tr_all, te = (folds != k) & (folds != v), folds == v, folds != k, folds == k
        fit_prep = (lambda m: Standardiser().fit(X[m])) if prep is None else \
            (lambda m: prep(X[m], None if groups is None else groups[m], y[m]))
        st = fit_prep(tr_in)
        ref = reference_A(ref_A[k], X, y, groups, tr_in, va, st, seed, k, device) if select != "mae" else None
        trials = []
        for i, s in enumerate(grid_settings(grid)):
            s = {**fixed, **s}
            _, vp, _ = fit_head(st(X[tr_in]), y[tr_in], s, s["max_epochs"], seed + 1000 * k + i, device,
                                st(X[va]), None if groups is None else groups[tr_in], y[tr_in])
            stats = [{"mae": float(np.abs(p - y[va]).mean())} if select == "mae"
                     else selection_stats_v3(p, y[va], groups[va], ref) for p in vp]
            best = min(range(len(stats)), key=lambda e: rank_key(stats[e], select))
            trials.append({"setting": {k2: s[k2] for k2 in grid}, "best_epoch": best + 1,
                           "val_mae": stats[best]["mae"], "_key": rank_key(stats[best], select),
                           **({} if select == "mae" else {"val_worst_group_tpr": stats[best]["worst_group_tpr"],
                                                         "val_feasible": stats[best]["feasible"]})})
        win = min(trials, key=lambda t: t["_key"])
        s = {**fixed, **win["setting"]}
        st = fit_prep(tr_all)
        predict_fn, _, info = fit_head(st(X[tr_all]), y[tr_all], s, win["best_epoch"], seed + 1000 * k, device,
                                       None, None if groups is None else groups[tr_all], y[tr_all])
        pred[te] = predict_fn(st(X[te]))
        rec = {"test_fold": k, "chosen": win["setting"], "epochs": win["best_epoch"],
               "inner_val_mae": win["val_mae"], "test_mae": float(np.abs(pred[te] - y[te]).mean()),
               "refit_info": info, "seconds": round(time.time() - t0, 1)}
        if select != "mae":
            rec["inner_val_worst_group_tpr"] = win["val_worst_group_tpr"]
            rec["reference_A_mae_diff"] = ref["mae_diff_vs_A_record"]
        if fixed.get("leakage_probe"):
            rec["leakage_probe"] = leakage_fold(X, y, groups, tr_all, te, st, predict_fn, ref_A[k], seed, k,
                                                device, probe_out)
        record.append(rec)
        log(f"fold {k}: chose {win['setting']} x {win['best_epoch']} epochs, test MAE {rec['test_mae']:.2f} "
            f"({rec['seconds']:.0f} s)")
    return pred, record

## 6. Experiments

### 6.1 Model A (baseline)
1. **I wonder:** does an age head trained on UTKFace's own ages, on frozen CLIP features, keep the zero-shot gap?
2. **I predict, because:** a gap still appears, probably smaller than zero-shot CLIP's 10 points, because the head learns from UTKFace's own age labels.
3. **To check, I will change** nothing (baseline) **and keep** the folds, the 8-setting grid and the selection by validation MAE fixed for every later model.

In [ ]:
GITHUB = "https://raw.githubusercontent.com/isaacmutuma/robust_child_filtering_model/main/results"
def committed(rel):
    """My committed result for comparison (aggregate numbers only)."""
    with urllib.request.urlopen(f"{GITHUB}/{rel}") as r:
        return json.loads(r.read())

d_ref = None
def evaluate(name, pred):
    d = prepare(frame(df.id.values, groups, ages, pred))
    global d_ref, plan
    if d_ref is None:
        d_ref, plan = d, BootPlan(d)
    rep = model_report(d, plan)
    g = rep["tpr_gap_W_minus_B"]
    print(f"{name}: W-B gap {g['est']:.3f} [{g['ci'][0]:.3f}, {g['ci'][1]:.3f}], MAE {rep['mae_overall']['est']:.2f}")
    return d, rep

RESULTS, RECORDS = {}, {}
if RUN["A"]:
    predA, RECORDS["A"] = oof_predictions(X, ages, folds, GRID_A, FIXED_A, DEVICE, groups=groups, select="mae")
    dA, RESULTS["A"] = evaluate("A", predA)
    refA = [{"chosen": r["chosen"], "epochs": r["epochs"], "inner_val_mae": r["inner_val_mae"]} for r in RECORDS["A"]]
    print("committed result:", round(committed("models/A.json")["report"]["tpr_gap_W_minus_B"]["est"], 3))

**After running** (committed run, `results/models/A.json`) — **I observed:** the White − Black gap is 11.0 points (95% CI 7.7 to 14.2), larger than zero-shot CLIP's 10.0. A catches 47.6% of Black and 80.9% of White 10–17-year-olds, and over-estimates Black minors' ages by 1.9 years more than White minors' (95% CI 1.2 to 2.7). Adult false alarms stay at most 0.6% in every group; MAE 4.44 years. **Interpretation:** training on UTKFace's own ages does not remove the gap; my prediction that it would shrink was wrong. The gap comes with over-estimation of Black minors' ages, which pushes teenagers over 18.

### 6.2 Model E (adversarial training on frozen CLIP features)
1. **I wonder:** can adversarial training on frozen CLIP features remove the race information that makes Black teenagers look older, and close the gap at 18?
2. **I predict, because:** a small effect; a probe will still recover race from its representation, because the bias is in CLIP's embeddings, which E cannot change.
3. **To check, I will change** only the training (a race adversary on the MLP's hidden layer, 5 adversary steps per MLP step, two targets, λ ∈ {0.1, 0.3, 1, 3}) **and keep** the folds, loss, budget, tuning rule and evaluation of A. Control: λ = 0 gives exactly model A.

In [ ]:
if RUN["E"]:
    probe_E = {}
    predE, RECORDS["E_inv"] = oof_predictions(X, ages, folds, GRID_E, FIXED_E, DEVICE, groups=groups,
                                              select="worst_group_v3", ref_A=refA, probe_out=probe_E)
    dE, RESULTS["E_inv"] = evaluate("E", predE)
    PAIRED = {"E_inv": paired_vs_A(dA, dE, plan)}
    LEAK_E = summarise(probe_E, groups)
    for n, r in LEAK_E["bal_acc"].items():
        print(f"race probe on {n:9s}: {r['est']:.3f} [{r['ci'][0]:.3f}, {r['ci'][1]:.3f}]  (chance 0.20)")

**After running** (committed run, `results/models/E_inv.json`) — **I observed:** the gap is 10.6 points (95% CI 7.1 to 13.7); the reduction against A is 0.4 points (95% CI −1.7 to 2.4, p = 0.36), at almost no cost (MAE +0.08 years). During training the adversary's race accuracy fell to chance (0.20–0.28 in four of five folds), yet a new linear probe reads race from E's hidden layer as well as from A's (balanced accuracy 0.816 vs 0.819; chance 0.20). The tuning rule picked the target 'make the adversary wrong' in every fold. **Interpretation:** the adversary was fooled, but race was not removed, as I predicted. **Not yet certain:** whether removing race would close the gap; this became follow-up 1.

### 6.3 Results so far

In [ ]:
def row(name, rep, paired=None):
    g, bg = rep["tpr_gap_W_minus_B"], rep["by_group"]
    out = {"model": name, "gap (points)": f"{100*g['est']:.1f} ({100*g['ci'][0]:.1f} to {100*g['ci'][1]:.1f})",
           "teen TPR Black / White": f"{bg['Black']['tpr_teen']['est']:.3f} / {bg['White']['tpr_teen']['est']:.3f}",
           "max adult FPR": f"{max(x['fpr_adult']['est'] for x in bg.values()):.3f}",
           "MAE": f"{rep['mae_overall']['est']:.2f}", "Black - White age error": f"{rep['age_err_gap_B_minus_W']['est']:+.2f}"}
    if paired:
        r = paired["gap_reduction_A_minus_M"]
        out["gap reduction vs A"] = f"{100*r['est']:+.1f} ({100*r['ci'][0]:+.1f} to {100*r['ci'][1]:+.1f}), p={r['p_one_sided']:.3f}"
    return out
display(pd.DataFrame([row(m, RESULTS[m], PAIRED.get(m) if m != "A" else None) for m in RESULTS]))

### 6.4 Figures

In [ ]:
PRED = {"A": predA, "E": predE}
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for name, pr in PRED.items():                                   # detection rate of minors by age
    for g, ls in [("White", "-"), ("Black", "--")]:
        m = groups == g
        tpr = [np.mean(pr[m & (ages == a)] < 18) for a in range(0, 18)]
        ax[0].plot(range(18), tpr, ls, label=f"{name} {g}")
ax[0].set(xlabel="true age", ylabel="share flagged as child", title="Minors flagged as children"); ax[0].legend(fontsize=8)
for g in ["White", "Black"]:                                     # predicted vs true age (model A)
    m = (groups == g) & (ages <= 30)
    mean_pred = [predA[m & (ages == a)].mean() for a in range(0, 31)]
    ax[1].plot(range(31), mean_pred, label=g)
ax[1].plot([0, 30], [0, 30], "k:", lw=1); ax[1].axhline(18, c="grey", lw=1)
ax[1].set(xlabel="true age", ylabel="mean predicted age (A)", title="Over-estimation near 18"); ax[1].legend()
for r in RECORDS["E_inv"]:                                       # E: adversary accuracy during training
    ax[2].plot(range(1, len(r["refit_info"]["adv_bal_acc_by_epoch"]) + 1), r["refit_info"]["adv_bal_acc_by_epoch"], label=f"fold {r['test_fold']}")
ax[2].axhline(0.2, ls="--", c="grey", lw=1); ax[2].set(xlabel="epoch", ylabel="adversary balanced accuracy", ylim=(0, 1), title="E: adversary is fooled")
ax[2].legend(fontsize=8); plt.tight_layout(); plt.show()

## 7. Fine-tuning inside CLIP (F1-1, F1-4, F2)

### 7.1 F1-1 and F1-4
1. **I wonder:** if CLIP's last vision blocks are allowed to change, does the White − Black gap at 18 narrow, and how deep does the change need to go?
2. **I predict, because:** fine-tuning has the best chance of narrowing the gap, because it can change how CLIP encodes age itself, which E and LEACE could not; but it may overfit, since there are only 126 Black teenagers.
3. **To check, I will change** only which weights can learn: LoRA (rank 8, α = 16) on the query, key and value projections of the last block (F1-1) or the last 4 blocks (F1-4), with the rest of CLIP frozen, **and keep** the folds, L1 loss, age head (started from A's head, so step 0 equals A), the tuning rule (epoch ≤ 10 chosen on the validation fold) and the evaluation of A and E.

### 7.2 F2 (the better depth + E's adversary)
1. **I wonder:** does the adversary help once CLIP itself can change?
2. **I predict, because:** F2 narrows the gap more than F1, because once CLIP's blocks can change, the adversary can actually remove race from the features instead of only being fooled, as it was on frozen features. *(Recorded 25 Nov 2026, before any F run.)*
3. **To check, I will change** only one thing against F1: add E's adversary (make it wrong, λ = 1, 5 adversary steps per step), **and keep** everything else; F2 against F1 is the ablation of the adversary.

**Implementation (my implementation).** open_clip's attention keeps q, k and v in one packed weight; `LoRAAttention` computes the same attention plus the low-rank update. Because blocks 1–20 are frozen, each image's tokens entering block 21 are computed once and stored (fp16, ~12.5 GB), and training runs only the last 4 blocks.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class LoRAAttention(nn.Module):
    """Self-attention (batch_first) = frozen nn.MultiheadAttention + LoRA on q/k/v."""

    def __init__(self, mha, rank=8, alpha=16, dropout=0.1):
        super().__init__()
        assert mha.batch_first and mha._qkv_same_embed_dim
        self.mha = mha
        for p in self.mha.parameters():
            p.requires_grad_(False)
        d = mha.embed_dim
        self.h, self.dh = mha.num_heads, d // mha.num_heads
        self.lora_A = nn.Parameter(torch.empty(rank, d))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.lora_B = nn.Parameter(torch.zeros(3 * d, rank))
        self.scale = alpha / rank
        self.drop = nn.Dropout(dropout)
        self.keep_cls_attention = False     # set True to record CLS-token attention maps
        self.cls_attention = None

    def forward(self, q_x, k_x=None, v_x=None, need_weights=False, attn_mask=None):
        x = q_x
        n, L, d = x.shape
        qkv = F.linear(x, self.mha.in_proj_weight, self.mha.in_proj_bias)
        qkv = qkv + self.scale * (self.drop(x) @ self.lora_A.t() @ self.lora_B.t())
        q, k, v = (t.view(n, L, self.h, self.dh).transpose(1, 2) for t in qkv.chunk(3, dim=-1))
        if self.keep_cls_attention:
            w = torch.softmax((q @ k.transpose(-2, -1)) / math.sqrt(self.dh), dim=-1)
            self.cls_attention = w[:, :, 0, :].detach()          # (n, heads, L): CLS query
            out = w @ v
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        out = out.transpose(1, 2).reshape(n, L, d)
        return self.mha.out_proj(out), None


def add_lora(visual, blocks, rank=8, alpha=16, dropout=0.1):
    """Freeze all of `visual`, then put LoRA into the attention of the given block indices."""
    for p in visual.parameters():
        p.requires_grad_(False)
    for i in blocks:
        blk = visual.transformer.resblocks[i]
        blk.attn = LoRAAttention(blk.attn, rank, alpha, dropout)
    return [p for i in blocks for p in (visual.transformer.resblocks[i].attn.lora_A,
                                        visual.transformer.resblocks[i].attn.lora_B)]


def tokens_before(visual, images, start):
    """Token sequence entering block `start` (patch + position embedding, ln_pre, blocks < start)."""
    x = visual._embeds(images)
    for blk in visual.transformer.resblocks[:start]:
        x = blk(x)
    return x


def finish(visual, tokens, start):
    """Blocks start.., then ln_post on the CLS token and the projection: the CLIP embedding."""
    x = tokens
    for blk in visual.transformer.resblocks[start:]:
        x = blk(x)
    pooled, _ = visual._pool(x)
    return pooled @ visual.proj if visual.proj is not None else pooled


def n_trainable(params):
    return int(sum(p.numel() for p in params))

In [ ]:
import copy
import time

import numpy as np
import torch
import torch.nn as nn


FIXED_F = {"rank": 8, "alpha": 16, "lora_dropout": 0.1, "lr": 1e-4, "weight_decay": 1e-4,
           "batch_size": 64, "max_epochs": 10,
           # F2 only (E's most-chosen setting)
           "adv_lambda": 1.0, "adv_target": "max_ce", "adv_hidden": 128, "adv_lr": 1e-2,
           "adv_steps": 5, "adv_weight_decay": 1e-4}


class FModel(nn.Module):
    def __init__(self, visual, start, lora_blocks, mu, sd, head, cfg):
        super().__init__()
        self.visual = copy.deepcopy(visual)
        self.lora_params = add_lora(self.visual, lora_blocks, cfg["rank"], cfg["alpha"], cfg["lora_dropout"])
        self.start = start
        self.register_buffer("mu", torch.as_tensor(mu, dtype=torch.float32))
        self.register_buffer("sd", torch.as_tensor(sd, dtype=torch.float32))
        self.head = copy.deepcopy(head)
        for p in self.head.parameters():
            p.requires_grad_(True)

    def standardised(self, tokens):
        e = finish(self.visual, tokens, self.start).float()
        e = e / e.norm(dim=1, keepdim=True)
        return (e - self.mu) / self.sd

    def z(self, tokens):
        return self.head[:2](self.standardised(tokens))

    def forward(self, tokens):
        return self.head(self.standardised(tokens)).squeeze(1)


def _batches(idx, bs, gen=None):
    order = idx[torch.randperm(len(idx), generator=gen).numpy()] if gen is not None else idx
    for b in range(0, len(order), bs):
        yield order[b:b + bs]


def _load(cache, rows, device):
    rows = np.sort(rows)   # memmap reads in order
    return rows, torch.from_numpy(np.asarray(cache[rows], dtype=np.float32)).to(device)


def predict_f(model, cache, rows, y_mu, y_sd, device, bs=256):
    model.eval()
    out = np.empty(len(rows))
    pos = {r: i for i, r in enumerate(rows)}
    with torch.no_grad():
        for b in _batches(np.asarray(rows), bs):
            srt, t = _load(cache, b, device)
            pred = model(t).float().cpu().numpy() * y_sd + y_mu
            for r, p_ in zip(srt, pred):
                out[pos[r]] = p_
    return out


def fit_f(cache, rows_tr, ytr, gtr, A_fn, mu, sd, visual, start, lora_blocks, cfg, epochs, seed, device,
          rows_val=None, adversary=False, log=print):
    """Train F on rows_tr of the token cache. A_fn: A's predict_fn for the same training rows
    (its .model is the head to start from, .y_mu/.y_sd its age scaling).
    Returns (predict_fn(rows) -> ages, per-epoch validation predictions or None, info)."""
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    model = FModel(visual, start, lora_blocks, mu, sd, A_fn.model.cpu(), cfg).to(device)
    params = model.lora_params + list(model.head.parameters())
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    y_mu, y_sd = A_fn.y_mu, A_fn.y_sd
    ystd = {r: (a - y_mu) / y_sd for r, a in zip(rows_tr, ytr)}
    loss_fn = nn.L1Loss()
    info = {"n_lora_params": int(sum(p.numel() for p in model.lora_params)), "epoch_seconds": []}
    if adversary:
        classes = sorted(set(np.asarray(gtr).tolist()))
        gid = {r: c for r, c in zip(rows_tr, np.searchsorted(classes, np.asarray(gtr)))}
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(seed + 7919)
            adv = nn.Sequential(nn.Linear(model.head[0].out_features, cfg["adv_hidden"]), nn.ReLU(),
                                nn.Linear(cfg["adv_hidden"], len(classes)))
        adv = adv.to(device)
        opt_adv = torch.optim.AdamW(adv.parameters(), lr=cfg["adv_lr"], weight_decay=cfg["adv_weight_decay"])
        ce_fn = nn.CrossEntropyLoss()
        info["adv_bal_acc_by_epoch"] = []
    use_amp = device == "cuda"
    val_preds = [] if rows_val is not None else None
    rows_tr = np.asarray(rows_tr)
    for ep in range(epochs):
        t0 = time.time()
        model.train()
        conf = np.zeros((5, 5), int) if adversary else None
        for b in _batches(rows_tr, cfg["batch_size"], gen):
            srt, t = _load(cache, b, device)
            y = torch.tensor([ystd[r] for r in srt], dtype=torch.float32, device=device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                if adversary:
                    z = model.z(t).float()
                    rid = torch.tensor([gid[r] for r in srt], device=device)
                    for _ in range(cfg["adv_steps"]):
                        opt_adv.zero_grad()
                        ce_fn(adv(z.detach()), rid).backward()
                        opt_adv.step()
                    out = model.head[2:](z).squeeze(1)
                    logits = adv(z)
                    loss = loss_fn(out, y) - cfg["adv_lambda"] * ce_fn(logits, rid)
                    np.add.at(conf, (rid.cpu().numpy(), logits.argmax(1).cpu().numpy()), 1)
                else:
                    loss = loss_fn(model(t).float(), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
        info["epoch_seconds"].append(round(time.time() - t0, 1))
        if adversary:
            info["adv_bal_acc_by_epoch"].append(float(np.mean(np.diag(conf) / np.maximum(conf.sum(1), 1))))
        if rows_val is not None:
            val_preds.append(predict_f(model, cache, rows_val, y_mu, y_sd, device))
        log(f"  epoch {ep + 1}/{epochs}: {info['epoch_seconds'][-1]:.0f} s")

    def predict_fn(rows):
        return predict_f(model, cache, rows, y_mu, y_sd, device)
    predict_fn.model = model
    return predict_fn, (np.stack(val_preds) if val_preds is not None else None), info

In [ ]:
START, F_CONFIGS = 20, {"F1_1": [23], "F1_4": [20, 21, 22, 23]}
tok_file = WORK / "tokens_b20.f16"
if RUN["F"]:
    clip_model, preprocess = load_clip(DEVICE)
    visual = clip_model.visual.float()
    if not tok_file.exists():
        cache = np.memmap(tok_file, np.float16, "w+", shape=(len(df), 257, visual.transformer.width))
        with torch.no_grad():
            for b in range(0, len(df), 64):
                x = torch.stack([preprocess(Image.open(q).convert("RGB")) for q in df.path[b:b + 64]]).to(DEVICE)
                cache[b:b + 64] = tokens_before(visual, x, START).cpu().numpy().astype(np.float16)
        cache.flush()
    cache = np.memmap(tok_file, np.float16, "r", shape=(len(df), 257, visual.transformer.width))

def run_F(blocks, adversary, seed=C.SEED):
    """Same protocol as E: epochs chosen on the validation fold (rule v3), refit on 4 folds."""
    pred, recs = np.full(len(ages), np.nan), []
    for k in range(C.N_FOLDS):
        v = (k + 1) % C.N_FOLDS
        tr_in, va, tr_all, te = (folds != k) & (folds != v), folds == v, folds != k, folds == k
        rA = refA[k]
        i = grid_settings(GRID_A).index({kk: rA["chosen"][kk] for kk in GRID_A})
        sA = {**FIXED_A, **rA["chosen"]}
        st_in, st_all = Standardiser().fit(X[tr_in]), Standardiser().fit(X[tr_all])
        ref_fn, vpA, _ = fit_head(st_in(X[tr_in]), ages[tr_in], sA, rA["epochs"], seed + 1000 * k + i, DEVICE, st_in(X[va]))
        ref = group_fpr_mae(vpA[-1], ages[va], groups[va]); ref["tpr"] = group_tpr(vpA[-1], ages[va], groups[va])
        refit_fn, _, _ = fit_head(st_all(X[tr_all]), ages[tr_all], sA, rA["epochs"], seed + 1000 * k, DEVICE)
        r_in, r_va, r_all, r_te = (np.flatnonzero(m) for m in (tr_in, va, tr_all, te))
        _, vp, _ = fit_f(cache, r_in, ages[r_in], groups[r_in], ref_fn, st_in.mu, st_in.sd, visual, START, blocks,
                         FIXED_F, FIXED_F["max_epochs"], seed + 1000 * k, DEVICE, rows_val=r_va, adversary=adversary)
        stats = [selection_stats_v3(p_, ages[r_va], groups[r_va], ref) for p_ in vp]
        best = min(range(len(stats)), key=lambda e: rank_key(stats[e], "v3"))
        fn, _, info = fit_f(cache, r_all, ages[r_all], groups[r_all], refit_fn, st_all.mu, st_all.sd, visual, START,
                            blocks, FIXED_F, best + 1, seed + 1000 * k, DEVICE, adversary=adversary)
        pred[r_te] = fn(r_te)
        recs.append({"test_fold": k, "chosen": {"lora_blocks": blocks}, "epochs": best + 1, "batch": FIXED_F["batch_size"],
                     "inner_val_worst_group_tpr": stats[best]["worst_group_tpr"],
                     "test_mae": float(np.abs(pred[r_te] - ages[r_te]).mean()), "seconds": sum(info["epoch_seconds"])})
        print(f"fold {k}: {best + 1} epochs, test MAE {recs[-1]['test_mae']:.2f}")
    return pred, recs

if RUN["F"]:
    for name, blocks in F_CONFIGS.items():
        predF, RECORDS[name] = run_F(blocks, adversary=False)
        dF, RESULTS[name] = evaluate(name, predF)
        PAIRED[name] = paired_vs_A(dA, dF, plan)
    best_f1 = max(F_CONFIGS, key=lambda n: np.mean([r["inner_val_worst_group_tpr"] for r in RECORDS[n]]))
    predF2, RECORDS["F2"] = run_F(F_CONFIGS[best_f1], adversary=True)
    dF2, RESULTS["F2"] = evaluate("F2", predF2)
    PAIRED["F2"] = paired_vs_A(dA, dF2, plan)
else:
    # Reuse the F run made with scripts/run_model_f.py (same code, same folds and seeds):
    # its per-image predictions are in F_OOF_DIR and its per-fold settings in results/models/<F>.json.
    F_OOF_DIR = Path.home() / "fwork"
    for name in ["F1_1", "F1_4", "F2"]:
        f = F_OOF_DIR / f"oof__{name}.csv"
        if not f.exists():
            print(f"{name}: no saved predictions at {f}; set RUN['F'] = True to train it here"); continue
        o = pd.read_csv(f, dtype={"id": str}).set_index("id").loc[df.id.values]
        assert (o.fold.values == folds).all(), "saved predictions use different folds"
        dF, RESULTS[name] = evaluate(name, o.pred_age.values)
        PAIRED[name] = paired_vs_A(dA, dF, plan)
        RECORDS[name] = [{**r, "chosen": {"lora_blocks": r["lora_blocks"]}, "batch": FIXED_F["batch_size"]}
                         for r in committed(f"models/{name}.json")["folds"]]
display(pd.DataFrame([row(m, RESULTS[m], PAIRED.get(m)) for m in RESULTS]))

**After running** — **I observed:** TODO (F results). **Interpretation:** TODO.

## 8. Re-check of the main comparison (3 seeds)

The key comparison (A against the best model) is repeated with seeds 1 and 2 (seed 0 is the main run): new random initialisation and batch order, same folds and rule. Set `BEST` to the best model after Section 7.

In [ ]:
BEST = "E_inv"          # set to the best model after Section 7 (e.g. "F1_4")
if RUN["recheck"]:
    rows_rc = []
    for seed in (0, 1, 2):
        pA_s, recA_s = oof_predictions(X, ages, folds, GRID_A, FIXED_A, DEVICE, seed=seed, groups=groups, select="mae")
        refA_s = [{"chosen": r["chosen"], "epochs": r["epochs"], "inner_val_mae": r["inner_val_mae"]} for r in recA_s]
        if BEST == "E_inv":
            pB_s, _ = oof_predictions(X, ages, folds, GRID_E, FIXED_E, DEVICE, seed=seed, groups=groups,
                                      select="worst_group_v3", ref_A=refA_s, probe_out={})
        else:
            refA_saved, refA = refA, refA_s
            pB_s, _ = run_F(F_CONFIGS.get(BEST, F_CONFIGS["F1_4"]), adversary=(BEST == "F2"), seed=seed)
            refA = refA_saved
        dA_s, dB_s = prepare(frame(df.id.values, groups, ages, pA_s)), prepare(frame(df.id.values, groups, ages, pB_s))
        r = paired_vs_A(dA_s, dB_s, plan)["gap_reduction_A_minus_M"]
        rows_rc.append({"seed": seed, "gap A": round(model_report(dA_s, plan)["tpr_gap_W_minus_B"]["est"], 3),
                        f"gap {BEST}": round(model_report(dB_s, plan)["tpr_gap_W_minus_B"]["est"], 3),
                        "reduction (95% CI)": f"{r['est']:+.3f} ({r['ci'][0]:+.3f} to {r['ci'][1]:+.3f})"})
    display(pd.DataFrame(rows_rc))

## 9. Follow-up questions

### 9.1 Follow-up 1
**Observation it came from:** E's adversary fell to chance while a new probe still read race at 0.82.
1. **I wonder:** if the adversary is pushed much harder, ignoring the cost limits, is race removed, and does the gap close?
2. **I predict, because:** the probe drops well below A's 0.82 and age error passes +0.5 years, but the gap shrinks far less than the race signal and the Black − White over-estimation stays above 0, because what makes Black teenagers look older is age cues in CLIP, not a separate race signal.
3. **To check, I will change** only the adversary's target and weight (8 fixed settings from λ = 0 to 10, 60 epochs each, no tuning) **and keep** everything else fixed.

In [ ]:
FU1_BASE = {"hidden": 512, "lr": 3e-4, "weight_decay": 1e-4}
FU1_SETTINGS = {"control": (0.0, "prior"), "prior-0.1": (0.1, "prior"), "prior-0.3": (0.3, "prior"),
                "prior-1": (1.0, "prior"), "prior-3": (3.0, "prior"), "prior-10": (10.0, "prior"),
                "max_ce-1": (1.0, "max_ce"), "max_ce-10": (10.0, "max_ce")}

def run_fixed(s, epochs=60, seed=C.SEED):
    """Out-of-fold predictions and race-probe predictions for one fixed setting (no tuning)."""
    pred, probe = np.full(len(ages), np.nan), np.full(len(ages), None, dtype=object)
    for k in range(C.N_FOLDS):
        tr, te = folds != k, folds == k
        st = Standardiser().fit(X[tr])
        pf, _, _ = fit_head(st(X[tr]), ages[tr], s, epochs, seed + 1000 * k, DEVICE, None, groups[tr], ages[tr])
        pred[te] = pf(st(X[te]))
        probe[te] = fit_predict_probe(pf.encode(st(X[tr])), groups[tr], pf.encode(st(X[te])), seed)
    return pred, probe

if RUN["followup1"]:
    fu1 = []
    for name, (lam, target) in FU1_SETTINGS.items():
        pred, probe = run_fixed({**FIXED_E, **FU1_BASE, "adv_lambda": lam, "adv_target": target})
        d, rep = evaluate(f"follow-up 1, {name}", pred)
        fu1.append({**row(name, rep, paired_vs_A(dA, d, plan)),
                    "race probe": round(summarise({"model": probe}, groups)["bal_acc"]["model"]["est"], 3)})
    display(pd.DataFrame(fu1))

**I observed:** in all 8 settings a new probe still reads race at 0.805–0.822 (threshold for 'race removed' 0.40), although the adversary itself was held near 0.50 or pushed to chance. Gaps range from 9.6 to 12.0 points and no setting narrows the gap significantly against A or against the λ = 0 control; the strongest settings raise MAE by 0.7–1.0 years and lower detection for both groups. **Interpretation:** adversarial training on a frozen head fools its adversary instead of removing race; my prediction that the probe would drop was wrong. **Not yet certain:** whether actually removing race would close the gap; this became follow-up 2.

### 9.2 Follow-up 2
**Observation it came from:** follow-up 1 could not remove race with an adversary at any strength.
1. **I wonder:** if linear race information is actually removed from CLIP's embeddings, does the White–Black age gap close, or does it stay wide?
2. **I predict, because:** the gap closes: once linear race information is removed, the gap falls to about 3 points or less with age error within +0.5 years, because linear race information is the real culprit.
3. **To check, I will change** only the input, with LEACE concept erasure fitted on each fold's training rows, **and keep** model A's exact procedure fixed.

LEACE (Belrose et al., 2023) edits the embeddings so that their covariance with race is exactly zero, which means no linear classifier can read race from them, while changing them as little as possible. My implementation follows the paper's closed form. Variant *band* first removes what each group's different age mix explains, so that age is not erased together with race.

In [ ]:
import numpy as np



def _onehot(labels, classes):
    labels = np.asarray(labels)
    return (labels[:, None] == np.asarray(classes)[None, :]).astype(np.float64)


class Leace:
    """Fit on training rows only; call to erase."""

    def fit(self, X, groups, ages=None, conditional=False, eps=1e-6):
        X = np.asarray(X, np.float64)
        self.classes = sorted(set(np.asarray(groups).tolist()))
        Z = _onehot(groups, self.classes)
        if conditional:   # race beyond the age-band mix
            bands = np.array([age_band(a) for a in ages])
            B = _onehot(bands, sorted(set(bands.tolist())))
            Z = Z - B @ np.linalg.lstsq(B, Z, rcond=None)[0]
        self.mu = X.mean(0)
        Xc, Zc = X - self.mu, Z - Z.mean(0)
        n = len(X)
        S = Xc.T @ Xc / n
        Sxz = Xc.T @ Zc / n
        ev, U = np.linalg.eigh(S)
        keep = ev > eps * ev.max()
        U, ev = U[:, keep], ev[keep]
        W = (U / np.sqrt(ev)) @ U.T             # Σ^{-1/2}
        W_pinv = (U * np.sqrt(ev)) @ U.T        # Σ^{1/2}
        A = W @ Sxz
        u, sv, _ = np.linalg.svd(A, full_matrices=False)
        u = u[:, sv > 1e-10 * max(sv.max(), 1e-300)]
        P = u @ u.T
        self.E = W_pinv @ P @ W                 # x' = x - E (x - mu)
        self.rank = int(u.shape[1])
        return self

    def __call__(self, X):
        X = np.asarray(X, np.float64)
        return (X - (X - self.mu) @ self.E.T).astype(np.float32)


class LeaceStandardiser:
    """Standardiser (L2 + z-score, training rows) followed by LEACE fitted on the same rows."""

    def __init__(self, conditional=False):
        self.conditional = conditional

    def fit(self, X, groups, ages):
        self.st = Standardiser().fit(X)
        self.leace = Leace().fit(self.st(X), groups, ages, conditional=self.conditional)
        return self

    def __call__(self, X):
        return self.leace(self.st(X))


def prep_factory(conditional=False):
    """For src.train.oof_predictions(prep=...)."""
    return lambda X, groups, ages: LeaceStandardiser(conditional).fit(X, groups, ages)


def mlp_probe(Ztr, gtr, Zte, seed=0):
    """Non-linear race probe (one hidden layer of 256)."""
    from sklearn.neural_network import MLPClassifier
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler().fit(Ztr)
    classes, codes = np.unique(np.asarray(gtr), return_inverse=True)
    clf = MLPClassifier(hidden_layer_sizes=(256,), early_stopping=True, max_iter=200, random_state=seed)
    clf.fit(sc.transform(Ztr), codes)
    return classes[clf.predict(sc.transform(Zte))]

if RUN["followup2"]:
    fu2 = []
    for name, cond in {"band": True, "plain": False}.items():
        pred, rec = oof_predictions(X, ages, folds, GRID_A, FIXED_A, DEVICE, groups=groups, select="mae",
                                    prep=prep_factory(cond))
        d, rep = evaluate(f"LEACE {name}", pred)
        lin, mlp = np.full(len(ages), None, dtype=object), np.full(len(ages), None, dtype=object)
        for k in range(C.N_FOLDS):
            tr, te = folds != k, folds == k
            le = LeaceStandardiser(cond).fit(X[tr], groups[tr], ages[tr])
            lin[te] = fit_predict_probe(le(X[tr]), groups[tr], le(X[te]), C.SEED)
            mlp[te] = mlp_probe(le(X[tr]), groups[tr], le(X[te]), C.SEED)
        pr = summarise({"linear": lin, "mlp": mlp}, groups)["bal_acc"]
        fu2.append({**row(f"LEACE {name}", rep, paired_vs_A(dA, d, plan)),
                    "linear probe": round(pr["linear"]["est"], 3), "MLP probe": round(pr["mlp"]["est"], 3)})
    display(pd.DataFrame(fu2))

**I observed:** LEACE removed linear race information completely (linear probe 0.200, exactly chance) at no cost in age error (MAE −0.03 years). The gap stayed at 10.0 points (95% CI 6.7 to 13.2; reduction 0.9 points, CI −1.1 to 2.9) and Black minors were still over-estimated by 1.9 years more than White minors. **Interpretation:** the bias is not carried by a linear race direction in CLIP's embeddings; my prediction that the gap would close was wrong. **Not yet certain:** race is still readable non-linearly (MLP probe 0.81) and the age head partly rebuilds it (0.60), so this test cannot separate a non-linear race signal from age cues that differ between groups. Changing CLIP itself (F1, F2) is the next test.

## 10. Real web photos (ICCWD): overall check and demo

ICCWD has no age or race labels, so it cannot measure the gap. It checks whether a model trained on face crops still finds children in ordinary web photos (child / no-child labels by the dataset authors). The final models are trained on all of UTKFace with A's most frequently chosen setting and the median chosen number of epochs. The images were downloaded to Google Drive with the authors' script (notebook 09); this section runs on Colab. The demo shows a few clearly harmless scenes as they are, with no captions or links.

In [ ]:
RUN_ICCWD = False
DEMO_ROWS = []          # row numbers of a few clearly harmless ICCWD scenes, picked by me
if RUN_ICCWD:
    from google.colab import drive
    drive.mount("/content/drive")
    IC = Path("/content/drive/MyDrive/robust_child_filtering_model/iccwd")
    ic = pd.read_csv(hf_hub_download("amcretu/iccwd", "image_caption_children_in_the_wild_dataset.csv", repo_type="dataset",
                                     revision="9f76b200c40a8a88c19dc17af8b842581f10f5f5"))
    ic["file"] = [IC / "images" / f"{i}.jpg" for i in range(len(ic))]
    ic = ic[ic.file.map(Path.exists) & ic.label.isin(["Final_Child", "Final_NoChild"])].reset_index(drop=True)
    clip_model, preprocess = load_clip(DEVICE)
    Xi = []
    with torch.no_grad():
        for b in range(0, len(ic), 64):
            x = torch.stack([preprocess(Image.open(f).convert("RGB")) for f in ic.file[b:b + 64]]).to(DEVICE)
            Xi.append(clip_model.encode_image(x).float().cpu().numpy())
    Xi = np.concatenate(Xi)
    from collections import Counter
    chosen = Counter(json.dumps(r["chosen"], sort_keys=True) for r in refA).most_common(1)[0][0]
    sA = {**FIXED_A, **json.loads(chosen)}
    epochs_A = int(np.median([r["epochs"] for r in refA]))
    st_full = Standardiser().fit(X)
    A_full, _, _ = fit_head(st_full(X), ages, sA, epochs_A, C.SEED, DEVICE)
    ic["pred_A"] = A_full(st_full(Xi))
    is_child = ic.label == "Final_Child"
    print(f"ICCWD (n={len(ic)}): A flags {np.mean(ic.pred_A[is_child] < 18):.3f} of photos with a child (TPR) "
          f"and {np.mean(ic.pred_A[~is_child] < 18):.3f} of photos without one (FPR)")
    for i in DEMO_ROWS:                                         # demo: a few clearly harmless scenes, shown as they are
        plt.figure(figsize=(4, 4)); plt.imshow(Image.open(ic.file[i]).convert("RGB")); plt.axis("off")
        plt.title(f"label: {'child' if is_child[i] else 'no child'} | A: age {ic.pred_A[i]:.0f} -> "
                  f"{'child' if ic.pred_A[i] < 18 else 'adult'}"); plt.show()

## 11. Failure cases, limits and conclusion

TODO.

## 12. Experiment log

In [ ]:
log_rows = []
for name, recs in RECORDS.items():
    for r in recs:
        log_rows.append({"run": name, "test fold": r["test_fold"], "chosen setting": r["chosen"], "epochs": r["epochs"],
                         "updates": r["epochs"] * math.ceil((folds != r["test_fold"]).sum() / r.get("batch", 256)),
                         "batch": r.get("batch", 256), "seed": f"{C.SEED} + 1000 x fold", "seconds": r["seconds"], "device": DEVICE})
display(pd.DataFrame(log_rows))

## 13. Sources and AI use

- Crețu et al., *Evaluating concept filtering defenses against child sexual abuse material generation by text-to-image models*, IEEE S&P 2026 (arXiv 2512.05707): motivation and the detectors' age-18 rule.
- Kireev, Crețu et al., *A manually annotated image-caption dataset for detecting children in the wild* (ICCWD), arXiv 2506.10117, and their download script (spring-epfl/iccwd): the real-photo check.
- Zhang, Song and Qi, *Age progression/regression by conditional adversarial autoencoder*, CVPR 2017: UTKFace.
- Radford et al., *Learning transferable visual models from natural language supervision* (CLIP), ICML 2021; Dosovitskiy et al., *An image is worth 16x16 words* (ViT), ICLR 2021: the backbone.
- Goodfellow et al., *Generative adversarial nets*, NeurIPS 2014; Ganin and Lempitsky, ICML 2015; Zhang, Lemoine and Mitchell, AIES 2018; Isola et al., pix2pix, CVPR 2017: adversarial training and the L1 + adversarial loss.
- Elazar and Goldberg, *Adversarial removal of demographic attributes from text data*, EMNLP 2018: why a fooled adversary does not mean race is gone.
- Belrose et al., *LEACE: perfect linear concept erasure in closed form*, NeurIPS 2023: follow-up 2.
- Hu et al., *LoRA: low-rank adaptation of large language models*, ICLR 2022; Zanella and Ben Ayed, *Low-rank few-shot adaptation of vision-language models*, CVPRW 2024: model F.
- Gokhale et al., arXiv 2104.01781 (2021): closest earlier work (adversarial debiasing of age estimates on UTKFace).
- Castleman and Korolova, *Adultification bias in LLMs and text-to-image models*, FAccT 2025: the same bias on the generation side.

**AI use:** an AI coding assistant (Claude) helped draft code, tests and text; I chose the question, wrote down my predictions before each run, ran the experiments and checked the results myself. *(Draft: confirm or edit.)*